# Composed Pipeline [Step 07.05 - Putting it together]

> **MLCourse - Agentic AI - LangGraph**

Time to build one system that uses every idea in this module at once:

```
                       PARENT: content pipeline
  START -> intake -> [ RESEARCH SUBGRAPH ]  -> [ WRITE+GRADE SUBGRAPH ] -> publish -> END
                       (isolated wrapper)        (isolated wrapper,
                        internally fans out       internally loops)
                        with Send)
```

- The **research subgraph** is attached with a wrapper (isolated state) and
  internally uses `Send` for runtime fan-out.
- The **write+grade subgraph** is attached with a wrapper too, and internally
  loops until the grade is acceptable or attempts run out.
- The parent knows none of that. It sees two boxes.

### What you'll learn

- How to layer `Send` *inside* a subgraph that is itself isolated from the parent.
- Where to put retry loops (inside the child, not the parent).
- How to observe the whole thing with `subgraphs=True` streaming.
- How to checkpoint a composed graph.

### Key takeaways

- Composition scales because each layer only knows its own contract.
- The wrapper functions are the documentation of the system.
- Rate-limit pacing matters: this notebook makes several model calls, all paced.

### Setup: environment, model factory, rate-limit-aware call helper


In [ ]:
import os                                  # environment variable access
import time                                # timing + backoff sleeps
from pathlib import Path                   # locating the track root
from dotenv import load_dotenv             # reads KEY=value pairs from .env

# Walk UP from the notebook folder until we find the track root `03_agentic_ai`,
# then load the (gitignored) .env that lives there. Every provider-touching
# notebook in this track uses exactly this block.
TRACK = Path.cwd()
while TRACK.name != "03_agentic_ai" and TRACK != TRACK.parent:
    TRACK = TRACK.parent
load_dotenv(TRACK / ".env")

GROQ_KEY = os.getenv("GROQ_API_KEY")       # never print this value
GROQ_MODEL = "qwen/qwen3.8-27b"            # fast hosted model, generous free tier
OLLAMA_MODEL = "llama3.1:8b"               # local fallback if Groq is unavailable


def make_llm(temperature: float = 0.0, max_tokens: int = 512):
    """Return a chat model. Groq first (fast, hosted); local Ollama as fallback.

    OpenAI is never used anywhere in this course.
    """
    if GROQ_KEY:
        from langchain_groq import ChatGroq
        return ChatGroq(model=GROQ_MODEL, api_key=GROQ_KEY,
                        temperature=temperature, max_tokens=max_tokens)
    from langchain_ollama import ChatOllama
    return ChatOllama(model=OLLAMA_MODEL, temperature=temperature)


def safe_invoke(model, messages, retries: int = 4, pause: float = 1.5):
    """Invoke a chat model with exponential backoff on rate limits (HTTP 429).

    Groq's free tier allows roughly 8000 tokens per minute. Teaching notebooks
    fire many small calls in a row, so a retry loop is not optional here.
    """
    delay = pause
    for attempt in range(retries):
        try:
            out = model.invoke(messages)
            time.sleep(pause)              # pace the next call politely
            return out
        except Exception as exc:
            if attempt == retries - 1:
                raise
            print("  [backoff] %s -- retrying in %.1fs" % (type(exc).__name__, delay))
            time.sleep(delay)
            delay *= 2                     # exponential backoff
    raise RuntimeError("unreachable")


print("Track root :", TRACK.name)
print("Provider   :", "Groq / " + GROQ_MODEL if GROQ_KEY else "Ollama / " + OLLAMA_MODEL)


### 1. Child A - the research subgraph (contains a `Send` fan-out)

Its vocabulary: `question` in, `notes` out. Internally it plans angles, fans out
one worker per angle, and merges.

In [2]:
from typing import Annotated, TypedDict
import operator, re, time
from langgraph.graph import StateGraph, START, END
from langgraph.types import Send


class ResearchState(TypedDict):
    question: str                              # INPUT
    angles: list                               # private
    notes: Annotated[list, operator.add]       # collected in parallel
    digest: str                                # OUTPUT


class AngleState(TypedDict):
    """Per-branch worker schema - what each Send payload carries."""
    question: str
    angle: str


plan_llm = make_llm(max_tokens=100)
note_llm = make_llm(max_tokens=110)
digest_llm = make_llm(max_tokens=200)


def plan_angles(state: ResearchState) -> dict:
    prompt = ("List 3 distinct angles worth researching for this question.\n"
              "Output ONLY the angles, one per line, no numbering.\n\n"
              "Question: " + state["question"])
    raw = safe_invoke(plan_llm, prompt).content
    angles = [re.sub(r"^[\s\-\*\d\.\)]+", "", ln).strip() for ln in raw.splitlines() if ln.strip()]
    angles = [a for a in angles if 3 < len(a) < 200][:3]
    return {"angles": angles}


def fan_out_angles(state: ResearchState):
    if not state["angles"]:
        raise ValueError("no angles planned")
    return [Send("study_angle", {"question": state["question"], "angle": a})
            for a in state["angles"]]


def study_angle(state: AngleState) -> dict:
    prompt = ("Answer in at most 2 sentences. Question: %s. Focus only on: %s"
              % (state["question"], state["angle"]))
    return {"notes": ["[%s] %s" % (state["angle"], safe_invoke(note_llm, prompt).content.strip())]}


def make_digest(state: ResearchState) -> dict:
    prompt = ("Condense these notes into a single tight paragraph of facts.\n\n"
              + "\n".join(state["notes"]))
    return {"digest": safe_invoke(digest_llm, prompt).content.strip()}


rg = StateGraph(ResearchState)
rg.add_node("plan_angles", plan_angles)
rg.add_node("study_angle", study_angle)
rg.add_node("make_digest", make_digest)
rg.add_edge(START, "plan_angles")
rg.add_conditional_edges("plan_angles", fan_out_angles, ["study_angle"])
rg.add_edge("study_angle", "make_digest")
rg.add_edge("make_digest", END)

research_sub = rg.compile()
print(research_sub.get_graph().draw_ascii())

 +-----------+   
 | __start__ |   
 +-----------+   
        *        
        *        
        *        
+-------------+  
| plan_angles |  
+-------------+  
        .        
        .        
        .        
+-------------+  
| study_angle |  
+-------------+  
        *        
        *        
        *        
+-------------+  
| make_digest |  
+-------------+  
        *        
        *        
        *        
  +---------+    
  | __end__ |    
  +---------+    


### 2. Child B - the write+grade subgraph (contains a retry loop)

Its vocabulary: `brief` + `facts` in, `copy` + `grade` + `attempts` out. Internally
it drafts, grades, and loops back on a weak grade - up to a cap.

Putting the loop **inside the child** is the point. The parent never has to know
that drafting is sometimes retried.

In [3]:
class WriteState(TypedDict):
    brief: str                 # INPUT
    facts: str                 # INPUT
    copy: str                  # OUTPUT
    grade: str                 # OUTPUT
    critique: str              # private
    attempts: int              # OUTPUT (useful telemetry)


draft_llm = make_llm(max_tokens=200)
grade_llm = make_llm(max_tokens=24)

MAX_ATTEMPTS = 2               # named constant, not a magic number


def draft(state: WriteState) -> dict:
    extra = ""
    if state.get("critique"):
        extra = "\nA previous attempt was rejected because: " + state["critique"]
    prompt = ("Write a 3-sentence marketing paragraph.\n"
              "Brief: %s\nFacts you must use: %s%s\n"
              "Output the paragraph only."
              % (state["brief"], state["facts"], extra))
    return {"copy": safe_invoke(draft_llm, prompt).content.strip(),
            "attempts": state.get("attempts", 0) + 1}


def grade(state: WriteState) -> dict:
    prompt = ("Reply with exactly one word - pass or fail - for whether this "
              "paragraph is clear, factual and 3 sentences long.\n\n" + state["copy"])
    word = safe_invoke(grade_llm, prompt).content.strip().lower()
    verdict = "pass" if "pass" in word else "fail"
    return {"grade": verdict,
            "critique": "" if verdict == "pass" else "not clear enough or wrong length"}


def should_retry(state: WriteState) -> str:
    if state["grade"] == "pass":
        return "done"
    if state["attempts"] >= MAX_ATTEMPTS:
        return "done"                       # give up gracefully, never loop forever
    return "retry"


wg = StateGraph(WriteState)
wg.add_node("draft", draft)
wg.add_node("grade", grade)
wg.add_edge(START, "draft")
wg.add_edge("draft", "grade")
wg.add_conditional_edges("grade", should_retry, {"retry": "draft", "done": END})

write_sub = wg.compile()
print(write_sub.get_graph().draw_ascii())

+-----------+  
| __start__ |  
+-----------+  
      *        
      *        
      *        
  +-------+    
  | draft |    
  +-------+    
      *        
      *        
      *        
  +-------+    
  | grade |    
  +-------+    
      .        
      .        
      .        
 +---------+   
 | __end__ |   
 +---------+   


### 3. The parent - two wrappers, its own vocabulary

The parent talks about `request`, `research_digest`, `article`, `quality`. Neither
child uses those names. Both are attached with adapter functions, exactly as taught
in notebook 03.

In [4]:
class PipelineState(TypedDict):
    request: str               # what the user asked for
    research_digest: str       # from child A
    article: str               # from child B
    quality: str               # from child B
    tries: int                 # from child B
    published: str             # parent's final output


def intake(state: PipelineState) -> dict:
    """Parent node: normalise the incoming request."""
    return {"request": state["request"].strip()}


def research_boundary(state: PipelineState) -> dict:
    """ADAPTER for child A. request -> question ; digest -> research_digest."""
    out = research_sub.invoke({
        "question": state["request"],
        "angles": [], "notes": [], "digest": "",
    })
    return {"research_digest": out["digest"]}


def write_boundary(state: PipelineState) -> dict:
    """ADAPTER for child B. request+digest -> brief+facts ; copy/grade -> article/quality."""
    out = write_sub.invoke({
        "brief": state["request"],
        "facts": state["research_digest"],
        "copy": "", "grade": "", "critique": "", "attempts": 0,
    })
    return {"article": out["copy"], "quality": out["grade"], "tries": out["attempts"]}


def publish(state: PipelineState) -> dict:
    """Parent node: final packaging."""
    badge = "OK" if state["quality"] == "pass" else "NEEDS REVIEW"
    return {"published": "[%s after %d attempt(s)]\n%s" % (badge, state["tries"], state["article"])}


pg = StateGraph(PipelineState)
pg.add_node("intake", intake)
pg.add_node("research", research_boundary)
pg.add_node("write", write_boundary)
pg.add_node("publish", publish)
pg.add_edge(START, "intake")
pg.add_edge("intake", "research")
pg.add_edge("research", "write")
pg.add_edge("write", "publish")
pg.add_edge("publish", END)

pipeline = pg.compile()
print(pipeline.get_graph().draw_ascii())

+-----------+  
| __start__ |  
+-----------+  
      *        
      *        
      *        
  +--------+   
  | intake |   
  +--------+   
      *        
      *        
      *        
+----------+   
| research |   
+----------+   
      *        
      *        
      *        
  +-------+    
  | write |    
  +-------+    
      *        
      *        
      *        
 +---------+   
 | publish |   
 +---------+   
      *        
      *        
      *        
 +---------+   
 | __end__ |   
 +---------+   


The parent diagram has four boxes. Behind them sit a runtime fan-out and a retry
loop, and the parent's state has six keys instead of the eighteen a flat version
would need.

### 4. Run it


In [5]:
t0 = time.time()
result = pipeline.invoke({
    "request": "why teams move from prompt chaining to graph-based agents",
    "research_digest": "", "article": "", "quality": "", "tries": 0, "published": "",
})
elapsed = time.time() - t0

print("=== RESEARCH DIGEST (child A) ===")
print(result["research_digest"])
print()
print("=== PUBLISHED (child B + parent) ===")
print(result["published"])
print()
print("wall clock: %.1fs" % elapsed)
print("parent state keys:", sorted(result.keys()))

=== RESEARCH DIGEST (child A) ===
Teams adopt graph-based agents to overcome the exponential state management complexity and error propagation risks of linear prompt chains, as graph structures explicitly model dependencies for scalable orchestration. This architecture enables parallel execution of independent tasks, significantly reducing cumulative latency and optimizing costs by avoiding redundant sequential processing, while also isolating failures within specific nodes to support dynamic routing and robust handling of non-linear workflows.

=== PUBLISHED (child B + parent) ===
[OK after 1 attempt(s)]
Teams are shifting from linear prompt chains to graph-based agents to overcome the exponential state management complexity and error propagation risks inherent in sequential workflows. By explicitly modeling dependencies, this architecture enables the parallel execution of independent tasks, significantly reducing cumulative latency and optimizing costs by avoiding redundant sequentia

Note what is **not** in the parent state: no `angles`, no `notes`, no `critique`,
no `brief`. Every child's private machinery stayed private, because the adapters
only returned what the parent's contract asked for.

### 5. Observing every layer

`stream(..., subgraphs=True)` shows the parent's steps. The children here are
invoked *inside* adapter functions, so they are separate runs - if you want their
internals in the same stream, embed them directly (shared state) or stream them
from within the adapter. This trade-off is worth knowing before you pick a mode.

In [6]:
print("--- parent-level updates ---")
for chunk in pipeline.stream(
        {"request": "benefits of typed agent state",
         "research_digest": "", "article": "", "quality": "", "tries": 0, "published": ""},
        stream_mode="updates"):
    for node, upd in chunk.items():
        preview = {k: (str(v)[:48] + "...") if isinstance(v, str) and len(str(v)) > 48 else v
                   for k, v in upd.items()}
        print("  %-9s %s" % (node, preview))

--- parent-level updates ---
  intake    {'request': 'benefits of typed agent state'}


  research  {'research_digest': 'Typed agent state enhances system reliability an...'}


  write     {'article': 'Typed agent state enhances system reliability an...', 'quality': 'pass', 'tries': 1}
  publish   {'published': '[OK after 1 attempt(s)]\nTyped agent state enhanc...'}


In [7]:
print("--- child A internals, streamed directly ---")
for chunk in research_sub.stream(
        {"question": "what makes agent evaluation hard",
         "angles": [], "notes": [], "digest": ""},
        stream_mode="updates"):
    for node, upd in chunk.items():
        if node == "study_angle":
            print("  branch  ->", upd["notes"][0][:70] + "...")
        elif node == "plan_angles":
            print("  planned ->", upd["angles"])
        else:
            print("  digest  ->", upd["digest"][:70] + "...")

--- child A internals, streamed directly ---


  planned -> ['The lack of ground truth for open-ended, multi-step tasks where success is defined by nuanced outcomes rather than binary correctness.', 'The non-deterministic and stateful nature of agent trajectories, which makes error attribution and causal analysis of failures extremely difficult.', 'The gap between static benchmark performance and dynamic real-world robustness, where agents must adapt to environmental changes and novel interactions not seen during training.']


  branch  -> [The non-deterministic and stateful nature of agent trajectories, whic...


  [backoff] RateLimitError -- retrying in 1.5s


  branch  -> [The lack of ground truth for open-ended, multi-step tasks where succe...


  branch  -> [The gap between static benchmark performance and dynamic real-world r...


  digest  -> Agent evaluation is hindered by the lack of definitive ground truth fo...


### 6. Checkpointing the composed pipeline

The parent compiles with a checkpointer like any other graph. Because the children
run inside adapters, the parent checkpoint stores the **contract** values only -
which is usually what you want to resume from.

In [8]:
from langgraph.checkpoint.memory import InMemorySaver

ck = pg.compile(checkpointer=InMemorySaver())
cfg = {"configurable": {"thread_id": "composed-1"}}
ck.invoke({"request": "when to use subgraphs",
           "research_digest": "", "article": "", "quality": "", "tries": 0, "published": ""}, cfg)

snap = ck.get_state(cfg)
print("stored keys:", sorted(snap.values.keys()))
print("next node  :", snap.next or "(finished)")
print()
print("history depth:", len(list(ck.get_state_history(cfg))))

stored keys: ['article', 'published', 'quality', 'request', 'research_digest', 'tries']
next node  : (finished)

history depth: 6


### 7. What you would change for production

- **Swap a child without touching the parent.** Replace `research_sub` with a
  version backed by a real search API; only `research_boundary` needs to know.
- **Version the children.** `research_sub_v2` can be A/B tested behind the same adapter.
- **Move the adapters into their own module.** They are the system's interface docs.
- **Add `max_concurrency`.** `pipeline.invoke(..., config={"max_concurrency": 4})`
  caps parallel `Send` branches - essential against a rate-limited provider.
- **Fail loudly on empty plans.** `fan_out_angles` already raises; keep that.

### Recap - the whole module in five bullets

1. Flat graphs break on key collisions, testability, reuse, and checkpoint granularity.
2. A compiled graph is a `Runnable`, so `add_node` can take it directly (shared state).
3. The boundary filters by **name** and silently drops non-matching keys - use a
   wrapper function to get an explicit, validated interface (isolated state).
4. `Send` creates branches at runtime; the payload is the worker's whole state and
   the collector key needs a reducer.
5. Layer them: isolated subgraphs on the outside, `Send` fan-out and retry loops inside.

### Next

**[08_advanced_reasoning_patterns](../08_advanced_reasoning_patterns/README.md)** -
now that you can compose graphs, use that power to build ReAct alternatives:
Reflexion, Plan-and-Execute, and ReWOO.